NOTEBOOK STRUCTURE

TITLE + OBJECTIVE
PROBLEM STATEMENT +
APPROACH

TASK 1 (markdown + code + output)
OBSERVATION

TASK 2 (markdown + code + output)
OBSERVATION

TASK 3 (markdown + code + output)
FINAL RESULT

# LLM Systems Engineer Assessment

**Name:   Sakthivel S M**

**B.Tech (Final Year) — Artificial Intelligence & Data Science**  
s.m.sakthivelofficial@gmail.com

---

##  Objective

The goal of this assessment is to demonstrate practical understanding of Large Language Models (LLMs) from a **systems engineering perspective**.  

This includes:
- Parsing model architectures  
- Fine-tuning using LoRA (efficient training)  
- Performing model composition (merging)  
- Evaluating improvements  
- Thinking about scalability and system design  

---

##  Problem Statement

Modern LLM systems are not just about using models — they require:

- Understanding internal architecture  
- Efficient training under hardware constraints  
- Combining models or adapters  
- Designing pipelines that can scale and evolve  

In this project, we aim to build a **complete mini LLM system pipeline**:

                   Parse → Fine-tune → Merge → Evaluate


---

## Approach & Plan

### 🔹 Task 1: Model Architecture Parsing
- Build a **recursive parser**
- Extract:
  - layers
  - attention blocks
  - MLPs
  - normalization layers
- Represent model as a **hierarchical structure**

---

### 🔹 Task 2: Fine-Tuning (LoRA)
- Use **TinyLlama-1.1B-Chat**
- Apply **LoRA (Low-Rank Adaptation)** for efficient training  
- Train on **IMDB dataset (sentiment classification)**  
- Use:
  - 4-bit quantization (memory efficient)
  - gradient checkpointing  

---

### 🔹 Task 3: Model Composition
- Merge LoRA adapter back into base model  
- Compare:
  - Base model performance  
  - Fine-tuned model performance  

---

### 🔹 Task 4: Analysis & Design Thinking
- Explain design decisions  
- Discuss scalability and limitations  
- Propose future improvements  

---

##  Challenges Expected

- Memory constraints (GPU limits)  
- Dataset compatibility issues  
- Handling large models efficiently  
- Ensuring correct training behavior (label masking)  

---

##  Expected Outcome

- Clear understanding of model structure  
- Improved performance after fine-tuning  
- A reusable and extensible LLM pipeline  

---

##  System Flow


Model → Parse Architecture
→ Fine-tune (LoRA)
→ Merge Adapter
→ Evaluate Performance


---



# ============================================================
# TASK 1: Model Architecture Parsing
# ============================================================

In this task, we build a recursive parser to analyze the internal structure of LLMs.

We extract:
- Attention layers
- MLP blocks
- Normalization layers
- Embeddings

The output is a hierarchical tree representation of the model.

In [ ]:


import torch
import torch.nn as nn
import json
from typing import Dict, Any
from transformers import AutoModelForCausalLM


# -------------------------------------------------------
# Count ONLY DIRECT parameters (no double counting)
# -------------------------------------------------------
def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters(recurse=False) if p.requires_grad)


# -------------------------------------------------------
# Classify module
# -------------------------------------------------------
def classify_module(module: nn.Module) -> str:
    name = module.__class__.__name__.lower()

    if "attention" in name or "attn" in name:
        return "attention"
    elif any(k in name for k in ["mlp", "feedforward", "ff", "intermediate"]):
        return "mlp"
    elif "norm" in name:
        return "normalization"
    elif "embedding" in name:
        return "embedding"
    elif "dropout" in name:
        return "regularization"
    elif any(k in name for k in ["encoder", "decoder", "layer", "block"]):
        return "block"
    else:
        return "other"


# -------------------------------------------------------
# Extract module details
# -------------------------------------------------------
def extract_details(module: nn.Module) -> Dict[str, Any]:
    details = {}

    if isinstance(module, nn.Linear):
        details["in_features"] = module.in_features
        details["out_features"] = module.out_features

    elif isinstance(module, nn.Embedding):
        details["num_embeddings"] = module.num_embeddings
        details["embedding_dim"] = module.embedding_dim

    elif isinstance(module, nn.LayerNorm):
        details["normalized_shape"] = list(module.normalized_shape)

    elif isinstance(module, nn.Conv2d):
        details["in_channels"] = module.in_channels
        details["out_channels"] = module.out_channels
        details["kernel_size"] = module.kernel_size

    return details


# -------------------------------------------------------
# Recursive parser
# -------------------------------------------------------
def parse_model(module: nn.Module, depth: int = 0) -> Dict:
    structure = {
        "type": module.__class__.__name__,
        "category": classify_module(module),
        "parameters": count_parameters(module),
        "depth": depth,
        "details": extract_details(module),
        "children": {}
    }

    for name, child in module.named_children():
        structure["children"][name] = parse_model(child, depth + 1)

    return structure


# -------------------------------------------------------
# Pretty print tree
# -------------------------------------------------------
def print_tree(structure: Dict, indent: int = 0):
    space = "  " * indent

    print(f"{space}+ {structure['type']} "
          f"[{structure['category']}] "
          f"({structure['parameters']:,} params)")

    for k, v in structure["details"].items():
        print(f"{space}    └─ {k}: {v}")

    for name, child in structure["children"].items():
        print(f"{space}  [{name}]")
        print_tree(child, indent + 2)


# -------------------------------------------------------
# Save JSON
# -------------------------------------------------------
def save_to_json(structure: Dict, filename: str):
    with open(filename, "w") as f:
        json.dump(structure, f, indent=2)
    print(f"Saved: {filename}")


# -------------------------------------------------------
# Main runner (SAFE VERSION)
# -------------------------------------------------------
def run_parser(model_name: str, save_name: str):

    print(f"\n{'='*60}")
    print(f"Loading: {model_name}")
    print(f"{'='*60}")

    with torch.no_grad():
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",          # auto CPU/GPU
            torch_dtype=torch.float16   # memory optimization
        )

    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {total:,}\n")

    parsed = parse_model(model)

    print("Architecture Tree:")
    print_tree(parsed)

    save_to_json(parsed, save_name)

    # Cleanup
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return parsed


# -------------------------------------------------------
# RUN MODELS
# -------------------------------------------------------

# GPT-2
gpt2_structure = run_parser(
    "gpt2",
    "gpt2_structure.json"
)

# TinyLlama
tinyllama_structure = run_parser(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "tinyllama_structure.json"
)

print("\n Task 1 complete. Both models parsed and saved.")


Loading: gpt2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total Trainable Parameters: 1,100,048,384

Architecture Tree:
+ LlamaForCausalLM [other] (0 params)
  [model]
    + LlamaModel [other] (0 params)
      [embed_tokens]
        + Embedding [embedding] (65,536,000 params)
            └─ num_embeddings: 32000
            └─ embedding_dim: 2048
      [layers]
        + ModuleList [other] (0 params)
          [0]
            + LlamaDecoderLayer [block] (0 params)
              [self_attn]
                + LlamaAttention [attention] (0 params)
                  [q_proj]
                    + Linear [other] (4,194,304 params)
                        └─ in_features: 2048
                        └─ out_features: 2048
                  [k_proj]
                    + Linear [other] (524,288 params)
                        └─ in_features: 2048
                        └─ out_features: 256
                  [v_proj]
                    + Linear [other] (524,288 params)
                        └─ in_features: 2048
                        └─ out_featu

## Observations (Task 1: Model Parsing)

- The parser successfully extracted a hierarchical structure of both models.
- Transformer architecture is clearly visible:
  - Repeating blocks containing attention + MLP layers
  - LayerNorm applied before/after key components
- GPT-2 has a relatively simpler architecture compared to TinyLlama.
- TinyLlama shows deeper structure with more modular blocks.
- Parameter distribution highlights that most parameters are concentrated in:
  - Attention projections
  - Feedforward (MLP) layers

### Result:
The recursive parser effectively represents model architecture as a tree, making it easy to inspect, compare, and extend for other models.

# ============================================================
# TASK 2: LoRA Fine-Tuning
# ============================================================

We fine-tune TinyLlama using LoRA for efficient training.

Key decisions:
- 4-bit quantization for memory efficiency
- Train only adapter layers (~0.4% parameters)
- Dataset: IMDB (sentiment classification)

Goal:
Adapt the model for classification without full retraining.

In [ ]:


!pip install unsloth datasets transformers accelerate bitsandbytes peft -q

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import PeftModel

# -------------------------------------------------------
# GPU Info
# -------------------------------------------------------
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -------------------------------------------------------
# Load model (4-bit)
# -------------------------------------------------------
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=256,
    load_in_4bit=True
)

# FIX: pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------------------------------------
# LoRA config
# -------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
)

# FIX: disable cache
model.config.use_cache = False

# -------------------------------------------------------
# Dataset
# -------------------------------------------------------
dataset = load_dataset("imdb")

train_data = dataset["train"].select(range(1000))
test_data  = dataset["test"].select(range(200))

# -------------------------------------------------------
# Only train on response tokens
# -------------------------------------------------------
def format_example(example):
    label = "positive" if example["label"] == 1 else "negative"

    prompt = f"""### Instruction:
Classify sentiment.

### Input:
{example['text'][:400]}

### Response:
Answer:"""

    full_text = prompt + " " + label

    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    # MASK prompt part
    prompt_tokens = tokenizer(
        prompt,
        truncation=True,
        max_length=256
    )["input_ids"]

    labels = tokens["input_ids"].copy()
    labels[:len(prompt_tokens)] = [-100] * len(prompt_tokens)

    tokens["labels"] = labels
    return tokens

train_data = train_data.map(format_example, remove_columns=["text", "label"])
test_data  = test_data.map(format_example, remove_columns=["text", "label"])

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# -------------------------------------------------------
# Training args
# -------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
)

# -------------------------------------------------------
# TRAIN
# -------------------------------------------------------
trainer.train()

# -------------------------------------------------------
# SAVE ADAPTER
# -------------------------------------------------------
model.save_pretrained("./tinyllama-lora-adapter")
tokenizer.save_pretrained("./tinyllama-lora-adapter")

# -------------------------------------------------------
# MERGE MODEL
# -------------------------------------------------------
del model
torch.cuda.empty_cache()

model_f16, tokenizer_f16 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=256,
    dtype=torch.float16,
    load_in_4bit=False
)

model_f16 = PeftModel.from_pretrained(model_f16, "./tinyllama-lora-adapter")
model_f16 = model_f16.merge_and_unload()

model_f16.save_pretrained("./tinyllama-merged")
tokenizer_f16.save_pretrained("./tinyllama-merged")

# -------------------------------------------------------
# EVALUATION (NEW — IMPORTANT)
# -------------------------------------------------------
def predict(text):
    prompt = f"""### Instruction:
Classify sentiment.

### Input:
{text[:400]}

### Response:
Answer:"""

    inputs = tokenizer_f16(prompt, return_tensors="pt").to(model_f16.device)

    with torch.no_grad():
        out = model_f16.generate(**inputs, max_new_tokens=5)

    decoded = tokenizer_f16.decode(out[0], skip_special_tokens=True).lower()

    if "positive" in decoded:
        return "positive"
    elif "negative" in decoded:
        return "negative"
    return "unknown"


correct = 0
for example in dataset["test"].select(range(50)):
    pred = predict(example["text"])
    label = "positive" if example["label"] == 1 else "negative"

    if pred == label:
        correct += 1

print(f"\nAccuracy on 50 samples: {correct}/50 = {correct/50:.2%}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/tinyllama-chat as a legacy tokenizer.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/di


Accuracy on 50 samples: 49/50 = 98.00%


## Observations (Task 2: Fine-Tuning with LoRA)

- LoRA enabled efficient training by updating only a small subset of parameters.
- Instead of full fine-tuning, only low-rank adapter layers were trained (~0.1–1% of total parameters).
- Training was memory-efficient due to:
  - 4-bit quantization
  - Gradient checkpointing
- The model quickly adapted to sentiment classification despite limited training data.
- Loss decreased steadily, indicating stable learning.

### Challenges Faced:
- Memory limitations in Colab GPU
- Dataset loading issues (pyarrow compatibility)
- Ensuring correct prompt formatting and label alignment

###  Result:
The fine-tuned model achieved strong learning on the target task with minimal resource usage, demonstrating the effectiveness of LoRA-based adaptation.

# ============================================================
# TASK 3: Model Composition & Evaluation
# ============================================================

We merge the LoRA adapter into the base model and evaluate:

- Base model performance
- Fine-tuned model performance

This demonstrates real improvement after fine-tuning.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import os, shutil

# -------------------------------------------------------
# Mount Drive
# NOTE: If running locally, comment out the drive.mount lines below.
# Set local_path = "./tinyllama-merged" pointing to your saved model folder.
# -------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

drive_path = "/content/drive/MyDrive/tinyllama-merged"
local_path = "./tinyllama-merged"

# Copy model safely
if os.path.exists(local_path):
    print("Local model already exists. Skipping copy.")
elif os.path.exists(drive_path):
    shutil.copytree(drive_path, local_path)
    print("Copied model from Drive.")
else:
    raise ValueError("Model not found. Please save from Task 2.")

# -------------------------------------------------------
# Dataset (REDUCED FOR SPEED)
# -------------------------------------------------------
dataset = load_dataset("imdb")
test_data = dataset["test"].select(range(50))

# -------------------------------------------------------
# Prompt
# -------------------------------------------------------
def make_prompt(text):
    return f"""### Instruction:
Classify the sentiment of the following movie review as positive or negative.

### Input:
{text[:400]}

### Response:
Answer:"""

# -------------------------------------------------------
# Label extraction
# -------------------------------------------------------
def extract_label(text):
    text = text.lower().strip()

    if "positive" in text:
        return 1
    elif "negative" in text:
        return 0
    return -1

# -------------------------------------------------------
# Prediction (FIXED)
# -------------------------------------------------------
def predict(model, tokenizer, text):
    inputs = tokenizer(
        make_prompt(text),
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return extract_label(decoded)

# -------------------------------------------------------
# Evaluation
# -------------------------------------------------------
def evaluate(model, tokenizer, label="Model"):
    correct, valid, skipped = 0, 0, 0

    for i, item in enumerate(test_data):
        pred = predict(model, tokenizer, item["text"])

        if pred == -1:
            skipped += 1
        else:
            valid += 1
            if pred == item["label"]:
                correct += 1

        if (i + 1) % 10 == 0:
            print(f"[{i+1}/50] running accuracy: {correct}/{valid}")

    acc = correct / valid if valid > 0 else 0

    print(f"\n{label}")
    print(f"  Correct  : {correct}/{valid}")
    print(f"  Skipped  : {skipped}")
    print(f"  Accuracy : {acc:.4f} ({acc*100:.1f}%)")

    return acc

# -------------------------------------------------------
# LOAD BASE MODEL
# -------------------------------------------------------
base_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_path)
tokenizer_base.pad_token = tokenizer_base.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

base_model.config.use_cache = True
base_model.eval()

print("\nEvaluating BASE MODEL...")
base_acc = evaluate(base_model, tokenizer_base, "Base Model")

del base_model
torch.cuda.empty_cache()

# -------------------------------------------------------
# LOAD FINE-TUNED MODEL
# -------------------------------------------------------
ft_path = "./tinyllama-merged"

tokenizer_ft = AutoTokenizer.from_pretrained(ft_path)
tokenizer_ft.pad_token = tokenizer_ft.eos_token

ft_model = AutoModelForCausalLM.from_pretrained(
    ft_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

ft_model.config.use_cache = True
ft_model.eval()

print("\nEvaluating FINE-TUNED MODEL...")
ft_acc = evaluate(ft_model, tokenizer_ft, "Fine-Tuned Model")

del ft_model
torch.cuda.empty_cache()

# -------------------------------------------------------
# FINAL RESULTS
# -------------------------------------------------------
delta = ft_acc - base_acc

print("\n================ FINAL RESULTS ================")
print(f"Base Accuracy       : {base_acc:.4f}")
print(f"Fine-tuned Accuracy : {ft_acc:.4f}")
print(f"Improvement         : {delta*100:+.2f}%")

if delta > 0.05:
    print("Strong improvement")
elif delta > 0.01:
    print(" Moderate improvement")
else:
    print("Small improvement (expected with small dataset)")

Mounted at /content/drive
Copied model from Drive.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Evaluating BASE MODEL...
[10/50] running accuracy: 3/10
[20/50] running accuracy: 5/20
[30/50] running accuracy: 6/30
[40/50] running accuracy: 9/40
[50/50] running accuracy: 13/50

Base Model
  Correct  : 13/50
  Skipped  : 0
  Accuracy : 0.2600 (26.0%)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Evaluating FINE-TUNED MODEL...
[10/50] running accuracy: 10/10
[20/50] running accuracy: 20/20
[30/50] running accuracy: 30/30
[40/50] running accuracy: 40/40
[50/50] running accuracy: 50/50

Fine-Tuned Model
  Correct  : 50/50
  Skipped  : 0
  Accuracy : 1.0000 (100.0%)

================ FINAL RESULTS ================
Base Accuracy       : 0.2600
Fine-tuned Accuracy : 1.0000
Improvement         : +74.00%
Strong improvement


## Observations (Task 3: Model Composition & Evaluation)

- The base TinyLlama model performed poorly (26% accuracy), indicating no inherent capability for sentiment classification.
- After merging the LoRA adapter, the model achieved perfect accuracy (100%) on the evaluation subset.
- This shows that:
  - The model successfully learned task-specific patterns
  - LoRA adaptation was highly effective

- The improvement (+74%) clearly demonstrates the impact of fine-tuning.

###  Note:
The evaluation was conducted on a small subset (50 samples), which may not fully reflect generalization performance. Larger evaluation would provide more robust validation.

---

## Final Results

| Model | Accuracy |
|------|--------|
| Base Model | 26% |
| Fine-tuned Model | 100% |
| Improvement | +74% |

---

### Result:
Model composition via LoRA merging significantly improved performance, validating the effectiveness of parameter-efficient fine-tuning.

## Overall Conclusion

This project demonstrates a complete LLM pipeline:

- Architecture understanding (Parser)
- Efficient training (LoRA)
- Model composition (Adapter merging)
- Quantitative evaluation

The results show that even small models can be significantly improved using efficient fine-tuning techniques, making them suitable for practical applications under limited resources.